## Introduction to Tensor Sketch

### 1. Kernel Machines

- Kernel machines are learning algorithms that rely on pairwise similarities between data points instead of working directly with feature coordinates.

- They don't operate on the data $x$ itself, but they operate on $K(x_i, x_j)$, a kernel function that measures similarity.

- Their can detect nonlinear structure in data using kernels, without explicitly mapping data to high-dimensional spaces.

### 2. Kernels

A kernel is a function that computes the inner product of two points in some (possibly huge or infinite) feature space without explicitly mapping the data there:
$$K(x_i, x_j) = \langle\phi(x), \phi(y) \rangle_\mathcal{H}$$
where:
- $\phi(x)$ = implicit nonlinear mapping to a high-dimensional space

- $\mathcal{H}$ = feature space (Hilbert space)

A kernel answers "How similar are x and y after a nonlinear transformation?" without computing $\phi(x)$.

### 3. Why Is a Kernel Useful?

- Nonlinear patterns become linear in feature space

- No need to explicitly compute high-dimensional features

- Universal function approximation

### 4. Kernel Trick

- Compute the inner product in feature space without computing the feature map — via the kernel function.

- This avoids computing the high-dimensional coordinates explicitly and supports various data types.

- We never compute feature map $\phi(x)$ when using the kernel trick.

- For example, for polynomial kernel (degree 2), 
$$k(x, y) = (x^{\top}y)^2$$

- Corresponding explicit map: 
$$\phi(x_1, x_2) = [x_1^2, \sqrt{2}x_1x_2, x_2^2]$$

- Theoretical $\phi(x)$: often infinite-dimensional.

### 5. What Is a Kernel Machine in Practice?

- Builds a model using only the Gram matrix $K_{ij} = K(x_i, x_j)$

- Does not need explicit coordinates

- Learn discision boundaries or regression functions using these similarities

### 6. Gram Matrix

Gram matrix = matrix of inner products between samples

Given a dataset with vectors:
$$X = \{x_1, x_2,\ldots, x_n \}$$
the Gram matrix is an $n \times n$ matrix where:
$$G_{ij} = \langle x_i, x_j \rangle = x_i^{\top}x_j $$
$$G = XX^{\top}$$
That means:
- each entry is the inner product between two samples

- the matrix describes similarity between all pairs of samples

Many machine learning algorithms do not use the raw data — they only use pairwise similarities.

### 7. What Kernel Machines Become Slow (Scalability Problem)

The full Gram matrix is: $n \times n$
- Times: $O(n^3)$

- Memory: $O(n^2)$

This is why Random Fourier Features, Count Sketch, and Tensor Sketch, were invented to approximate kernels without building the Gram matrix.

### 8. Random Feature Mapping

**Modified from Pham & Pagh (2013)**

Given any two points 
$$
x = \{x_1, \ldots, x_d \}, \quad
y = \{y_1, \ldots, y_d \} \in \mathbb{R}^d, 
$$

a kernel method uses a feature map

$$
\phi : \mathbb{R}^d \mapsto \mathcal{F}
$$
where $\mathcal{F}$ is a (usually high-dimensional or infinite-dimensional) feature space.

In this space, the similarity between $x$ and $y$ is computed by inner product:

$$
\langle \phi(x), \phi(y) \rangle_{\mathcal{F}} = \kappa(x, y)
$$
where $\kappa()$ is an easily computable kernel (e.g., polynomial, RBF).

**Random Feature Mapping**

A random feature map
$$
f : \mathbb{R}^d \mapsto \mathbb{R}^D
$$
is a computable, explicit, low-dimensional mapping that approximates the kernel.

It's designed such that 
$$
\mathbb{E}\!\left[ \langle f(x), f(y) \rangle \right]
    = \langle \phi(x), \phi(y) \rangle_{\mathcal{F}}
    = \kappa(x, y).
$$
We can transform data from the original data space into a low-dimensional explicit random feature space and use any linear learning algorithm to find non-linear data relations.

### 9. Linear vs. Nonlinear Problems

A linear model in the original space:
$$f(x) = w^Tx$$
cannot capture nonlinear relationships.

But if we map the input to a higher-dimensional feature space:
$$\phi : \mathcal{X} \to \mathcal{V}$$
then we can use a linear model in $\mathcal{V}$ to represent extremely nonlinear functions in $\mathcal{X}$

Example (degree-2 polynomial features):
$$\phi(x_1, x_2) = [x_1^2, \sqrt{2}x_1x_2, x_2^2]$$
A linear classifier in $\phi(x)$ becomes a quadratic classifier in the original space.

In [1]:
import numpy as np

def poly_kernel(x, y):
    return (np.dot(x, y))**2

def explicit_phi(x):
    return np.array([
        x[0]**2,
        np.sqrt(2) * x[0] * x[1],
        x[1]**2
    ])

x = np.array([2, 3])
y = np.array([4, 1])

print("Kernel:", poly_kernel(x, y))
print("⟨φ(x), φ(y)⟩:", np.dot(explicit_phi(x), explicit_phi(y)))

Kernel: 121
⟨φ(x), φ(y)⟩: 121.0


### 10. Types of Multiplications

#### 1.)  Inner Product (Dot Product)

The inner product (or dot product) between two vectors measures similarity and produces a scalar.

Given two vectors $\mathbf{x}, \mathbf{y} \in \mathbb{R}^n$:

$$\langle \mathbf{x}, \mathbf{y} \rangle = \sum_{i=1}^{n} x_i\, y_i$$

Or in matrix form:
$$\mathbf{x}^{\top}\mathbf{y}$$


In [2]:
import numpy as np

x = np.array([1, 2, 3])
y = np.array([4, 5, 6])

inner = np.dot(x, y)
print(inner) 

32


#### 2.)  Element-wise Multiplication (Hadamard Product)

Element-wise multiplication multiplies each corresponding element of two vectors or matrices of the same shape.

For vectors:
$$(\mathbf{x} \odot \mathbf{y})_i = x_i y_i$$

In [3]:
import numpy as np

x = np.array([1, 2, 3])
y = np.array([4, 5, 6])

elem = x * y
print(elem)

[ 4 10 18]


#### 3.)  Tensor Product
##### 3.1  Outer Product
The outer product of two vectors produces a matrix:

$$
\mathbf{x} \otimes \mathbf{y} =
\begin{bmatrix}
x_1 y_1 & x_1 y_2 & \cdots & x_1 y_n \\
x_2 y_1 & x_2 y_2 & \cdots & x_2 y_n \\
\vdots  & \vdots  & \ddots & \vdots  \\
x_m y_1 & x_m y_2 & \cdots & x_m y_n
\end{bmatrix}
$$

Given an integer $p$, we consider a $p$-level tensor product $\Omega^{p} : \mathbb{R}^{d} \mapsto \mathbb{R}^{d^{p}}$ given by 
$$
x \;\mapsto\; x^{(p)}
    = \underbrace{x \otimes x \otimes \cdots \otimes x}_{p \text{ times}} .
$$

LEMMA: Given any pair of points $x$, $y$ and an integer $p$, we have:
$$
\langle x^{(p)}, y^{(p)} \rangle = \langle x, y \rangle^p 
$$

In [4]:
import numpy as np

x = np.array([1, 2])
y = np.array([3, 4, 5])

outer = np.outer(x, y)
print(outer)

[[ 3  4  5]
 [ 6  8 10]]


##### 3.2  Kronecker Product
Given matrices $A \in \mathbb{R}^{m \times n}$ and $B \in \mathbb{R}^{p \times q}$:

$$
A \otimes B =
\begin{bmatrix}
a_{11} B & a_{12} B & \cdots & a_{1n} B \\
a_{21} B & a_{22} B & \cdots & a_{2n} B \\
\vdots   & \vdots   & \ddots & \vdots   \\
a_{m1} B & a_{m2} B & \cdots & a_{mn} B
\end{bmatrix}
$$
Resulting matrix has size: $(mp) \times (nq)$

In [5]:
import numpy as np

A = np.array([[1, 2],
              [3, 4]])

B = np.array([[0, 5],
              [6, 7]])

kron = np.kron(A, B)
print(kron)

[[ 0  5  0 10]
 [ 6  7 12 14]
 [ 0 15  0 20]
 [18 21 24 28]]


### 11. Hash Functions for Count Sketch

Count Sketch needs two things:

1. Hash function: $h(i) \mapsto \{0, 1, \ldots, m-1\}$

2. Sign Hash: $s(i) \in \{-1, +1\}$

In [6]:
import numpy as np

np.random.seed(0)

n = 5      # input dimension
m = 4      # sketch dimension

# Random hash bucket for each coordinate
h = np.random.randint(0, m, size=n) # hash buckets

# Random Rademacher signs
s = np.random.choice([-1, 1], size=n) # random signs

print("h:", h) 
print("s:", s)

h: [0 3 1 0 3]
s: [1 1 1 1 1]


Feature 0 → bucket 0, sign +1

Feature 1 → bucket 3, sign +1

Feature 2 → bucket 1, sign +1

### 12. Count Sketch Projection

Count Sketch vector: $$\mathrm{CS}(x)[j] = \sum_{i:\,h(i)=j} s(i)x_i$$

Count Sketch might provide better performance than traditional random projections in applications dealing with sparse vectors.

In [8]:
import numpy as np

n = 5
m = 4

h = np.array([0, 3, 1, 0, 3])      # hash buckets
s = np.array([ 1, -1, -1,  1,  1]) # random signs

x = np.array([ 2,  3, -1,  4,  5])

def countsketch(x, h, s, m):
    sk = np.zeros(m)
    for i in range(len(x)):
        sk[h[i]] += s[i] * x[i]
    return sk

cs = countsketch(x, h, s, m)

print("x =", x)
print("h =", h)
print("s =", s)
print("CountSketch(x) =", cs)

x = [ 2  3 -1  4  5]
h = [0 3 1 0 3]
s = [ 1 -1 -1  1  1]
CountSketch(x) = [6. 1. 0. 2.]


### 13. Tensor Sketch (Polynomial Kernel Approximation)

TensorSketch approximates: $$\phi(x) = x^{\otimes p}$$
but without forming the massive tensor.

It uses:
- CountSketch on each copy of $x$
- FFT → multiply in Fourier space → inverse FFT

In [14]:
from numpy.fft import fft, ifft

def tensorsketch_degree2(x, h1, s1, h2, s2, m):
    # Step 1: CountSketch each copy
    cs1 = countsketch(x, h1, s1, m)
    cs2 = countsketch(x, h2, s2, m)
    
    # Step 2: FFT, pointwise multiply, inverse FFT
    return np.real(ifft(fft(cs1) * fft(cs2)))

# Random setup
np.random.seed(1)
n = 5
m = 8

h1, s1 = np.random.randint(0, m, n), np.random.choice([-1,1], n)
h2, s2 = np.random.randint(0, m, n), np.random.choice([-1,1], n)

x = np.array([1.0, 2.0, 3.0, -1.0, 0.5])

sketch = tensorsketch_degree2(x, h1, s1, h2, s2, m)
print("TensorSketch(x) =", sketch)

TensorSketch(x) = [10.    3.5   1.5  -0.5  -8.75 -6.5  -2.5   5.5 ]


### 14. Computing True $\phi(x)$ vs Tensor Sketch Approximation

Compare:
$$\langle \tilde{\phi}(x), \tilde{\phi}(y) \rangle \approx (x^\top y)^2$$


In [15]:
# Two random vectors
x = np.random.randn(5)
y = np.random.randn(5)

# True kernel value
true_k = (np.dot(x, y))**2

# TensorSketch approximations
sk_x = tensorsketch_degree2(x, h1, s1, h2, s2, m)
sk_y = tensorsketch_degree2(y, h1, s1, h2, s2, m)

approx_k = np.dot(sk_x, sk_y)

print("True kernel =", true_k)
print("Approx kernel =", approx_k)


True kernel = 28.75692279756577
Approx kernel = 17.106311986458735


### 15. Exploring scikit-learn utility

In [9]:
from sklearn.kernel_approximation import PolynomialCountSketch
import numpy as np

In [ ]:
# Create dataset
X = np.array([[1, 2], [2, 1], [3, 0], [0, 3]], dtype=float)
print(X.shape) # (samples, features)

(4, 2)


In [ ]:
# Apply Tensor Sketch
# degree p
# n_components = sketch dimension
ps = PolynomialCountSketch(
    degree=2,           # approximates (x^T y)^2
    n_components=8,     # compressed feature dim
    random_state=0
)

X_sketch = ps.fit_transform(X)

print("TensorSketch Feature Map:\n", X_sketch)
print("Shape:", X_sketch.shape) # Now each sample has 8 new features.

TensorSketch Feature Map:
 [[ 2.22044605e-16  1.00000000e+00 -1.11022302e-16  0.00000000e+00
   4.00000000e+00  1.66533454e-16  1.11022302e-16  4.00000000e+00]
 [ 2.22044605e-16  4.00000000e+00  0.00000000e+00  1.66533454e-16
   4.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]
 [ 0.00000000e+00  9.00000000e+00  0.00000000e+00  3.14018492e-16
   0.00000000e+00  0.00000000e+00  0.00000000e+00 -3.14018492e-16]
 [ 0.00000000e+00  3.14018492e-16  0.00000000e+00  0.00000000e+00
   0.00000000e+00 -3.14018492e-16  0.00000000e+00  9.00000000e+00]]
Shape: (4, 8)


We are approximating the polynomial feature map:
$$\phi(x) = x^{\otimes 2}$$
For degree 2 and input dim = 2 (original feature), 
the true dimension is 
$$\binom{d+p-1}{p} = \binom{2+2-1}{2} = 3$$

But instead of outputting the exact 3-dimensional feature, Tensor Sketch outputs an 8-dimensional compressed version.

Tensor Sketch does not compute the true polynomial features; it computes a random projection of the high-degree tensor.

TensorSketch is built for high-dimensional input.

Larger `n_components` → lower variance → better approximation

In [ ]:
# True polynomial kernel matrix
true_K = (X @ X.T)**2       # Kernel Gram matrix always uses X @ X.T
print("True Kernel:", true_K)

# Approximated kernel using Tensor Sketch
approx_K = X_sketch @ X_sketch.T
print("Approx Kernel:\n", approx_K)

True Kernel: [[25. 16.  9. 36.]
 [16. 25. 36.  9.]
 [ 9. 36. 81.  0.]
 [36.  9.  0. 81.]]
Approx Kernel:
 [[3.3000000e+01 2.4000000e+01 9.0000000e+00 3.6000000e+01]
 [2.4000000e+01 3.3000000e+01 3.6000000e+01 9.0000000e+00]
 [9.0000000e+00 3.6000000e+01 8.1000000e+01 1.4791142e-31]
 [3.6000000e+01 9.0000000e+00 1.4791142e-31 8.1000000e+01]]


Why increase dimensionality?
- To encode nonlinear interactions (polynomial kernel features) in a compressed but expressive form.

### 16. Application in ML

In [20]:
from sklearn.kernel_approximation import PolynomialCountSketch
from sklearn.linear_model import SGDClassifier

# ------------------------------------------------------
# 1. Tiny toy dataset (XOR-like pattern)
# ------------------------------------------------------
# Points:
#   [0,0] and [1,1] belong to class 0  → these are on the diagonal
#   [1,0] and [0,1] belong to class 1  → off-diagonal points
#
# This dataset is NOT linearly separable in its original form.
# A linear model cannot separate these classes in 2D space.
# Kernel methods or nonlinear features are required.
# ------------------------------------------------------

X = [[0, 0],
     [1, 1],
     [1, 0],
     [0, 1]]

y = [0, 0, 1, 1]   # labels

# ------------------------------------------------------
# 2. PolynomialCountSketch → TensorSketch for polynomial kernels
# ------------------------------------------------------
# We choose a polynomial kernel of degree 3:
#
#   K(x, y) = (x^T y)^3
#
# Instead of computing the kernel matrix or explicitly building the
# full polynomial feature map φ(x), which would be expensive for
# large feature dimensions, TensorSketch gives an APPROXIMATION:
#
#   φ̃(x)  ~  φ(x)
#
# with a chosen sketch dimension (default = 100). The idea is:
#   - convert nonlinear features into a compressed linear representation
#   - allow linear models (like SGDClassifier) to behave nonlinearly
#   - scale better than kernel SVMs (no O(n^2) kernel matrix)
# ------------------------------------------------------

ps = PolynomialCountSketch(
    degree=3,        # approximate (x^T y)^3 kernel
    random_state=1   # reproducibility
)

# ------------------------------------------------------
# 3. Transform X into nonlinear (sketched) features
# ------------------------------------------------------
# X_features now contains the randomized polynomial features.
# Shape will be (4, n_components), with n_components ≈ 100 by default.
# ------------------------------------------------------

X_features = ps.fit_transform(X)

# ------------------------------------------------------
# 4. Train a linear classifier on the sketched nonlinear features
# ------------------------------------------------------
# SGDClassifier is a fast linear model.
# But because we fed nonlinear features, this becomes effectively
# a NONLINEAR classifier without computing explicit polynomial features.
# ------------------------------------------------------

clf = SGDClassifier(max_iter=10, tol=1e-3)
clf.fit(X_features, y)

# ------------------------------------------------------
# 5. Evaluate performance
# ------------------------------------------------------
# Since the dataset is tiny and TensorSketch approximates well,
# the classifier typically achieves perfect accuracy (1.0).
# ------------------------------------------------------

score = clf.score(X_features, y)
score


/opt/homebrew/Caskroom/miniconda/base/lib/python3.12/site-packages/sklearn/linear_model/_stochastic_gradient.py:723: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


1.0

### 17. Theory: Tensor Sketch (Pham & Pagh, 2013)

#### 1.) The Convolution of Count Sketches (Pagh, 2013)
We want a Count Sketch of the outer product:
$$x \otimes x \in \mathbb{R}^{d^{2}}$$
But computing the outer product explicitly costs $O(d^2)$, which is too big.

Instead:
- Compute two separate Count Sketches: $C^{(1)}x, C^{(2)}x $

- Combine them using FFT-based convolution to obtain $C(x \otimes x)$

Define: 
$$
H(i, j) = \big(h_1(i) + h_2(j)\big) \bmod D,
\qquad
S(i, j) = s_1(i)\, s_2(j)
$$
Where: 
- $h_1, h_2: [d] \to [D]$ are 2-wise independent hash function

- $s_1, s_2: [d] \to \{+1, -1\}$

This gurantees:
- $H$ and $S$ are valid Count Sketch hash functions over $d^2$ elements

- The resulting sketch is unbiased

Next, we represent each Count Sketch vector as a polynomial:
$$
P^{(1)}_{x}(\omega)
    = \sum_{i=1}^{d} s_{1}(i)\, x_i\, \omega^{\,h_{1}(i)},
\qquad
P^{(2)}_{x}(\omega)
    = \sum_{i=1}^{d} s_{2}(i)\, x_i\, \omega^{\,h_{2}(i)}
$$

**Remark**

A length-$D$ vector $v = (v_0, v_1, \ldots, v_{D-1})$ can be encoded as a polynomial:
$$
P_v(\omega) = \sum_{j=0}^{D-1} v_j\omega^j
$$ 
So:
- Vector view: $C^{(1)}x = [v_0, v_1, \ldots, v_{D-1}]$

- Polynomial view: $P^{(1)}_{x}(\omega) = v_0 + v_1\omega + \ldots + v_{D-1}\omega^{D-1}$

Then, we can compute the degree-($D$-1) polynomial for $Cx^{(2)}$ using hash functions $H$ and $S$:
$$ 
\begin{aligned}
P_{x^{(2)}}(\omega)
&= \sum_{i=1}^{d} \sum_{j=1}^{d}
    S(i,j)\, x_i x_j \, \omega^{H(i,j)}  \\
&= \mathrm{FFT}^{-1}\!\left[
    \mathrm{FFT}(P^{(1)}_{x})
    \odot
    \mathrm{FFT}(P^{(2)}_{x})
\right]
\end{aligned}
$$
where $ \odot$ is the element-wise product

In other words, the Count Sketch $Cx^{(2)}$ of $x \otimes x$ can be efficiently computed by Count Sketches $Cx^{(1)}$, $Cx^{(2)}$ in $O(d+D\log{D})$ times.

#### 2.) Tensor Sketch


The goal is to approximate the degree-p polynomial kernel 
$$
\kappa(x, y) = \langle x, y\rangle^p
$$
without explicitly constructing the huge polynomial feature space, with dimension $\binom{d+p-1}{p}$.

To approximate $\langle x, y\rangle^p$, we compute Count Sketches of: 
$$
x^{(p)}
    = \underbrace{x \otimes x \otimes \cdots \otimes x}_{p \text{ times}} 
$$
We define composite hash functions:
$$
H(i_1, \ldots, i_p)
    = \left( \sum_{k=1}^{p} h_k(i_k) \right) \bmod D,
\qquad
S(i_1, \ldots, i_p)
    = \prod_{k=1}^{p} s_k(i_k)
$$
where:
- $H: [d^p] \mapsto [D]$ and $S: [d^p] \mapsto \{+1, -1\}$

- $h_k$ are 2-wise independent

- $s_k$ are 4-wise independent (needed for variance bounds)

Tensor Sketch provides:
- A random feature map $f(x) \in \mathbb{R}^D$

- Such that $$\mathbb{E}[\langle f(x), f(y) \rangle] = \kappa(x, y)$$
- Computed in near-linear time: $$O(n(d+D\log{D}))$$

$$
\textbf{TensorSketch}(\mathbf{x}, p, D)
$$

$$
\begin{aligned}
&\textbf{Input:}~ \mathbf{x} \in \mathbb{R}^d,\; D,\; p>1, \\
&\quad h_1,\ldots,h_p : [d]\to[D],\;
     s_1,\ldots,s_p : [d]\to\{-1,1\} \\
&\textbf{Output:}~ f(\mathbf{x})\in\mathbb{R}^D \\
\\[-0.5em]
&1.\;\; \text{For } i=1,\ldots,p,\; \text{construct CountSketch } C_i \mathbf{x}. \\
&2.\;\; \widehat{C}_i \mathbf{x} \leftarrow \mathrm{FFT}(C_i \mathbf{x}). \\
&3.\;\; \widehat{C} \mathbf{x} \leftarrow \widehat{C}_1 \mathbf{x} \circ \cdots \circ \widehat{C}_p \mathbf{x}. \\
&4.\;\; f(\mathbf{x}) \leftarrow \mathrm{FFT}^{-1}(\widehat{C} \mathbf{x}). \\
&5.\;\; \text{return } f(\mathbf{x}).
\end{aligned}
$$

**Explanation**

Given vector $ \mathbf{x} \in \mathbb{R}^d$, sketch size $D$, polynomial degree $p$:

1. Compute p Count Sketches
$$
C_i x = \mathrm{CountSketch}(x; h_i, s_i), \qquad i = 1,\ldots,p
$$

2. Apply FFT to each sketch
$$
\widehat{C_i x} = \mathrm{FFT}(C_i x)
$$

3. Multiply them element-wise (convolution in polynomial domain)
$$
\widehat{C x} = \widehat{C_1 x} \;\circ\; \cdots \;\circ\; \widehat{C_p x}
$$

4. Inverse FFT to obtain final sketch 
$$
f(x) = \mathrm{FFT}^{-1}(\widehat{C x})
$$

5. Output
$$
f(x) \in \mathbb{R}^{D}, \qquad \mathbb{E}\!\left[\langle f(x), f(y) \rangle\right] = \langle x, y \rangle^{p}
$$

**Complexity**
1. Per vector:
$$
O(d+D\log{D})
$$

2. For $n$ vectors:
$$
O(n(d+D\log{D}))
$$

3. Hash function memory:
$$
O(1)
$$